In [1]:
from langchain_ibm import ChatWatsonx
from langchain_core.prompts import (
    PromptTemplate,
    ChatPromptTemplate,
    MessagesPlaceholder,
)

from langchain_core.output_parsers import (
    StrOutputParser,
    JsonOutputParser,
    PydanticOutputParser,
)

from dotenv import load_dotenv
import os

from langchain_community.document_loaders import PyPDFLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import FAISS

from langchain_ibm import WatsonxEmbeddings
from langchain_chroma import Chroma

from pydantic import BaseModel, Field
from langchain_core.runnables import RunnablePassthrough

import re

from pprint import pprint

load_dotenv()

apikey=  os.getenv("WATSONX_API_KEY")
project_id = os.getenv("WATSONX_PROJECT_ID")
watson_ai_url = os.getenv("WATSONX_URL")

watson_llm = ChatWatsonx(
    apikey=apikey,
    project_id=project_id,
    watsonx_url=watson_ai_url,
    model_id="ibm/granite-4-h-small",
    max_tokens=2000,
    params = {
        "temperature": 0
    }
)

watson_embedding = WatsonxEmbeddings(
    model_id="ibm/granite-embedding-278m-multilingual",
    url=f"{watson_ai_url}",
    api_key=f"{apikey}",
    project_id=f"{project_id}",
    params={"temperature": 0},
)

# 문서 로드
pdf_path = "./data/2018 서비스·집단 분쟁조정 사례집.pdf"

loader = PyPDFLoader(pdf_path)
docs = loader.load()
print(len(docs)) # 예상 값 : 200
pprint(docs[0].page_content)
pprint(docs[0].metadata)

# 전처리 - 정규식을 이용한 내용 추출
pattern = r"사\s*례\s*\d+.*?사건번호.*?결정일자.*?\d{4}\.\s?\d{1,2}\.\s?\d{1,2}\."

target_text = "".join(docs[10].page_content)
split_text = re.findall(pattern, target_text, re.DOTALL)

parts = []
if split_text: # 패턴이 존재할때 패턴을 기준으로 문서 분리
    parts = re.split(pattern, target_text)

    # 사례번호
    case_no = re.findall(r"례\s?(\d+)\s?사건번호", split_text[0])
    print(case_no[0] if case_no else "사례번호 미검출")
else:
    print("패턴 매칭 결과 없음")

# 사건 번호, 결정일자 추출
class CaseMetadata(BaseModel):
    case_number:str = Field(description="사건번호 예: 2018일나565")
    decision_date:str = Field(description="결정일자 예: 2018. 8. 7")


metadata_prompt = PromptTemplate.from_template(
    """\
        다음은 분쟁 조정 사례에 대한 텍스트입니다.

        - case_number : 사건번호
        - decision_date : 결정일자

        반드시 JSON 으로만 반환하세요
        {case_text}
    """
)

structed_llm = watson_llm.with_structured_output(CaseMetadata)
chain = metadata_prompt | structed_llm

if split_text:
    case_metadata = chain.invoke({
        "case_text" : split_text[0]
    })

    print(case_metadata)
    print(dict(case_metadata))


# 주 문 ~~~ (탐색용: parts 가 분할됐고 두 번째 조각이 있을 때만)
if len(parts) > 1:
    pprint(parts[1])

    m = re.search(r"주 문\n", parts[1])
    if m:
        title = parts[1][:m.span()[0]].replace("\n", "").strip()      # 주문 앞부분 = 제목
        content = parts[1][m.span()[0]:].strip()                       # 주문 이후 = 내용
        pprint(title)
        pprint(content)

# 사례 번호, 사건 번호, 결정일자, 제목은 meta데이터 추가
docs = []
case_metadata = {}

# 사건이 시작되는 페이지 ~ 마지막에서 -2 페이지 까지 반복
for doc in docs[10:-2]:
    split_text = re.findall(pattern, "".join(doc.page_content), re.DOTALL)
    if split_text:  # 새 사례가 시작되는 페이지
        case_metadata = {}

        # 사례 번호 추출
        case_no = re.findall(r"례\s?(\d+)\s?사건번호", split_text[0])
        case_metadata["case_no"] = case_no[0] if case_no else ""

        # 패턴 기준으로 텍스트 분할
        parts = re.split(pattern, "".join(doc.page_content))

        if len(parts) > 1 and re.search(r"주 문\n", parts[1]):
            span = re.search(r"주 문\n", parts[1]).span()
            # 제목 추출 (주문 앞부분)
            case_metadata["title"] = parts[1][:span[0]].replace("\n", "").strip()
            # 내용 추출 후 기존 내용 업데이트 (주문 이후)
            doc.page_content = parts[1][span[0]:].strip()
        else:
            case_metadata["title"] = ""

        # 사건 번호, 결정 일자 추출
        i = 0
        while i < 10:
            try:
                response = chain.invoke({"case_text": split_text[0]})
                for k, v in dict(response).items():
                    case_metadata[k] = v.replace("\n", "").replace(" ", "")
                break
            except Exception:
                i += 1
                continue

        doc.metadata.update(case_metadata)
        docs.append(doc)
    else:
        doc.metadata.update(case_metadata)
        docs.append(doc)

len(docs)

c:\source\ollama\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\soldesk\AppData\Local\Temp\ipykernel_10492\1399059002.py:17: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


200
'소비자분쟁조정위원회\n2018\n서비스·집단 \n분쟁조정 사례집'
{'author': 'PC_A2',
 'creationdate': '2019-06-05T11:33:24+09:00',
 'creator': 'PScript5.dll Version 5.2.2',
 'moddate': '2019-06-05T11:58:31+09:00',
 'page': 0,
 'page_label': '1',
 'producer': 'Acrobat Distiller 9.0.0 (Windows)',
 'source': './data/2018 서비스·집단 분쟁조정 사례집.pdf',
 'title': '<32303139303630355FBCD2BAF1C0DABFF820BBE7B7CAC1FD5BBCADBAF1BDBA5D5FB3BBC1F65FC6EDC1FDBABB76657231312E687770>',
 'total_pages': 200}
01
case_number='2018일나565' decision_date='2018. 8. 7.'
{'case_number': '2018일나565', 'decision_date': '2018. 8. 7.'}
('\n'
 '세탁 후 갑피 마모 및 경화된 가죽 \n'
 '운동화에 대한 손해배상 요구\n'
 '주 문\n'
 '1. 신청인은 2018. 10. 16.까지 피신청인에게 이 사건 제품(제품명 : ○○○○ 가죽 \n'
 '운동화, 색상 : 흰색) 1켤레를 반환한다. \n'
 '2. 피신청인은 신청인으로부터 제1항 제품을 반환받음과 동시에 신청인에게 71,000원\n'
 '을 지급한다.\n'
 '이 유\n'
 '1. 기초사실\n'
 '가. 신청인은 2017. 6. 6. 가죽 운동화(제품명 : ○○○○ 가죽 운동화, 색상 : 흰색, \n'
 '이하 ‘이 사건 제품’) 1켤레를 160,200원에 구매하여 착화하였고, 2018. 1. 10. \n'
 '피신청인에게 이 사건 제품의 세탁을 의뢰(세탁비 4,000원)하였는데 수령 후 갑피 \n'
 '마모 및 

0

In [4]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

split_docs = splitter.split_documents(docs)

print(split_docs)

[]


In [ ]:
# 마침표 뒤에 나오는 줄바꿈 문자를 그대로 두고 나머지 줄바꿈 문자만 제거
pprint(split_docs[18].page_content)

text = split_docs[18].page_content

text = re.sub(r"(?<!\.)\n"," ", text)
pprint(text)

('주 문\n'
 '1. 신청인은 2018. 10. 16.까지 피신청인에게 이 사건 제품(제품명 : ○○○○ 가죽 \n'
 '운동화, 색상 : 흰색) 1켤레를 반환한다. \n'
 '2. 피신청인은 신청인으로부터 제1항 제품을 반환받음과 동시에 신청인에게 71,000원\n'
 '을 지급한다.\n'
 '이 유\n'
 '1. 기초사실\n'
 '가. 신청인은 2017. 6. 6. 가죽 운동화(제품명 : ○○○○ 가죽 운동화, 색상 : 흰색, \n'
 '이하 ‘이 사건 제품’) 1켤레를 160,200원에 구매하여 착화하였고, 2018. 1. 10. \n'
 '피신청인에게 이 사건 제품의 세탁을 의뢰(세탁비 4,000원)하였는데 수령 후 갑피 \n'
 '마모 및 경화된 사실(이하 ‘이 사건 현상’)을 확인하여 피신청인이 재세탁을 하였\n'
 '으나, 이후에도 경화현상만 다소 개선될 뿐 갑피 마모 현상이 개선되지 않아 피신\n'
 '청인에게 손해배상(세탁비 환급 포함)을 요구하였으며, 피신청인은 세탁과실이 없\n'
 '다는 이유로 이를 거부하였다.\n'
 '나. 한국소비자원 신발제품심의위원회 심의 결과는 다음과 같다.')
('주 문 1. 신청인은 2018. 10. 16.까지 피신청인에게 이 사건 제품(제품명 : ○○○○ 가죽  운동화, 색상 : 흰색) 1켤레를 '
 '반환한다.  2. 피신청인은 신청인으로부터 제1항 제품을 반환받음과 동시에 신청인에게 71,000원 을 지급한다.\n'
 '이 유 1. 기초사실 가. 신청인은 2017. 6. 6. 가죽 운동화(제품명 : ○○○○ 가죽 운동화, 색상 : 흰색,  이하 ‘이 사건 '
 '제품’) 1켤레를 160,200원에 구매하여 착화하였고, 2018. 1. 10.  피신청인에게 이 사건 제품의 세탁을 의뢰(세탁비 '
 '4,000원)하였는데 수령 후 갑피  마모 및 경화된 사실(이하 ‘이 사건 현상’)을 확인하여 피신청인이 재세탁을 하였 으나, 이후에도 '
 '경화현상만 다소 개선될 뿐 갑피 마모 현상이 개선되지 않아 피신 청

In [ ]:
from copy import deepcopy

final_docs = []

for doc in split_docs:
    new_doc = deepcopy(doc)

    text = re.sub(r"(?<!\.)\n"," ", new_doc.page_content)

    new_doc.page_content = (
        f"### 이 사건은 '{new_doc.metadata['title']}에 대한 사례 입니다. \n\n"
        f"{text}"
    )

    final_docs.append(new_doc)